# 13.07 Hugging Face — 공급자 카탈로그

**`langchain-huggingface`** 한 패키지로 HF 의 4 가지 진입점을 다룬다 — **로컬 transformers** · **추론 엔드포인트(TGI)** · **Inference API** · **임베딩(SentenceTransformers)**.
다른 공급자와 달리 **API 키 없이 로컬 모델만으로도** 임베딩·추론을 돌릴 수 있다는 점이 특징.

## 학습 목표

- 한 패키지에서 끌어오는 클래스 매핑(ChatHuggingFace / HuggingFaceEndpoint / HuggingFacePipeline / HuggingFaceEmbeddings) 파악
- 4 가지 실행 모드 — local pipeline / TGI 컨테이너 / HF Inference API / Inference Endpoints (managed)
- 1순위 차별 기능: **TGI(Text Generation Inference) 자체 호스팅** — VPC 내 GPU 박스에 OpenAI 호환 서버 띄우기
- 한국어 임베딩 모델 1순위: `BAAI/bge-m3` (다국어, 8192 컨텍스트, dense+sparse+colbert 동시)

## 13.07.1 환경 설정

필요 패키지: `langchain-huggingface` + `sentence-transformers` (임베딩) + `transformers torch` (로컬 pipeline) + `huggingface_hub` (Endpoints).

```bash
uv pip install langchain-huggingface sentence-transformers
```

**키 없는 probe**: 로컬 임베딩 모델 한 번 다운받아 `embed_query("ping")` 가 끝까지 도는지 확인. 첫 실행은 모델 다운로드로 수 분 소요 가능.

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_huggingface import HuggingFaceEmbeddings

# 가장 작은 다국어 모델로 probe — 키 불필요, 모델만 캐시에 받아 둔다
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
v = emb.embed_query("ping")
print("dim:", len(v), "head:", v[:3])

## 13.07.2 공급자 제품군 전체 지도

| 카테고리 | 심볼 | 비고 |
|---------|------|------|
| Chat (Endpoints/TGI 래퍼) | `ChatHuggingFace` | `HuggingFaceEndpoint` 또는 `HuggingFacePipeline` 을 chat 인터페이스로 감쌈 |
| LLM (managed Endpoints/Inference API) | `HuggingFaceEndpoint` | URL 또는 repo_id 지정 |
| LLM (local) | `HuggingFacePipeline.from_model_id(...)` | `transformers.pipeline` 로컬 실행 |
| Embeddings (local SentenceTransformers) | `HuggingFaceEmbeddings` | `model_name="BAAI/bge-m3"` 등 |
| Embeddings (관리형 endpoint) | `HuggingFaceEndpointEmbeddings` | TEI(Text Embedding Inference) 서버 가리키기 |
| Datasets loader | `HuggingFaceDatasetLoader` (`langchain-community`) | HF Datasets 허브 → Document |

카테고리 노트북: 임베딩은 [`02_embeddings/06_huggingface.ipynb`](../02_embeddings/06_huggingface.ipynb).
Chat 모델 카탈로그에는 별도 HF 노트북이 없는 대신 본 13.07 에서 chat 진입점을 다룬다.

## 13.07.3 기본 사용 — embedding (로컬, 키 불필요)

다른 공급자 카탈로그가 chat 으로 시작하는 것과 달리 HF 는 **임베딩이 1차 진입점**이다 (가장 흔한 사용 패턴 + 키 불필요).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# 한국어 포함 다국어 RAG 의 1순위 추천 — bge-m3 (1024d, 8k 컨텍스트)
# 첫 실행은 ~2GB 다운로드로 수 분 소요 가능. 노트북 실행 시간 절감을 위해 작은 모델로 시연.
small = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
docs = small.embed_documents(["환불 정책", "배송 추적", "회원 탈퇴"])
print("count:", len(docs), "dim:", len(docs[0]))

## 13.07.4 공급자 특화 기능 — TGI 자체 호스팅

HF 의 1순위 차별 기능은 **Text Generation Inference (TGI)** 다. Hugging Face 가 만든 **GPU LLM 서빙 컨테이너**로,
OpenAI 호환 `/v1/chat/completions` 또는 native `/generate` 엔드포인트를 띄운다. 자체 인프라 안에서 오픈 모델을 돌리고 싶을 때 사실상 표준.

```bash
# GPU 서버에서 한 줄로 시작
docker run --gpus all -p 8080:80 \
  ghcr.io/huggingface/text-generation-inference:latest \
  --model-id meta-llama/Llama-3.3-70B-Instruct
```

LangChain 에서는 그 URL 을 `HuggingFaceEndpoint` 로 가리킨다.

```python
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    endpoint_url='http://gpu-box:8080',
    task='text-generation',
    max_new_tokens=512,
)
chat = ChatHuggingFace(llm=llm)
chat.invoke('한국어로 한 문장')
```

임베딩 측 자매 컨테이너는 **TEI (Text Embedding Inference)** — 같은 패턴으로 `HuggingFaceEndpointEmbeddings` 가 가리킨다.

또 다른 차별점: **HF Hub 의 100k+ 오픈 모델** — 라이선스 · 한국어 fine-tune · 도메인 특화 모델을 즉시 후보로 쓸 수 있다.

In [ ]:
# 시그니처만 검증 — 실제 TGI 서버가 필요하므로 호출은 주석으로
import inspect
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

print('HuggingFaceEndpoint fields head:', list(HuggingFaceEndpoint.model_fields)[:10])
print('ChatHuggingFace fields head:', list(ChatHuggingFace.model_fields)[:10])

## 13.07.5 가격 · 한도 · 실행 모드

| 모드 | 비용 | 지연 | 적합 |
|------|------|------|------|
| **Local pipeline** (`HuggingFacePipeline`) | 0 (전기·디스크) | CPU 시 수 초/토큰 | 프로토타입, 임베딩, 작은 모델 |
| **HF Inference API** (free tier) | 0 (rate-limited) | 수 초 cold start | 평가, 데모 — 프로덕션 X |
| **HF Inference Endpoints** (managed) | 시간당 GPU 요금 | 수 백 ms (warm) | 프로덕션, 자체 호스팅 부담 회피 |
| **TGI 자체 호스팅** | 본인 GPU | 수 십 ms (warm) | 데이터 거버넌스, VPC 내 추론, 대규모 처리량 |

- Embeddings 는 `bge-m3` 처럼 큰 모델도 단일 GPU/CPU 에서 충분 — 임베딩은 거의 항상 자체 호스팅이 비용 효율 우위.
- 라이선스 — 모델별로 상업 사용 가능 여부가 다르다. Llama 3 계열은 별도 라이선스 동의, Apache/MIT 모델은 자유.
- 리전 개념은 호스팅 측 책임 — Inference Endpoints 는 AWS/GCP 리전 선택 가능.

## 핵심 정리

- **embedding 1순위**: 다국어 RAG = `BAAI/bge-m3` (dense+sparse+colbert), 비용 우선 = `paraphrase-multilingual-MiniLM-L12-v2`
- **chat 1순위**: 자체 호스팅 = TGI + `HuggingFaceEndpoint`, 매니지드 = HF Inference Endpoints, 프로토타입 = `HuggingFacePipeline`
- **API 키 불필요**한 임베딩이 가장 흔한 사용 패턴 — 다른 공급자 chat 과 짝지어 쓴다
- **TGI/TEI 컨테이너**가 HF 의 사실상 표준 서빙 — OpenAI 호환 엔드포인트로 클라이언트 코드 재사용

## 다음

- [`02_embeddings/06_huggingface.ipynb`](../02_embeddings/06_huggingface.ipynb) — `HuggingFaceEmbeddings` 깊은 사용 + bge-m3 패턴
- [`01_chat_models/04_ollama.ipynb`](../01_chat_models/04_ollama.ipynb) — 비교: 로컬 LLM 의 또 다른 표준 (Ollama)

## 참고

- `langchain-huggingface` PyPI: https://pypi.org/project/langchain-huggingface/
- 공식 docs: https://docs.langchain.com/oss/python/integrations/providers/huggingface
- TGI: https://huggingface.co/docs/text-generation-inference/index
- TEI: https://huggingface.co/docs/text-embeddings-inference/index